In [1]:
import xarray as xr
import xcdat as xc
import icechunk as ic
from icechunk import Repository,s3_storage

# Initialize icechunk store
jaxa_gportal_url = "https://gportal.jaxa.jp/"

# 1. Open your repository
repo = Repository.open(s3_storage(
            bucket="airborne-smce-prod-user-bucket",
            prefix="JOIN/icechunk-stores/GCOM-W1-AMSR2-L3-SND"
        ),
        authorize_virtual_chunk_access={jaxa_gportal_url: ic.credentials.HttpAccess}
    )

# 2. Open a read-only session for the main branch
session = repo.readonly_session("main")

# 3. Read dataset
ds = xr.open_zarr(session.store, mask_and_scale=False)

# 4. Use xcdat to standardize the longitude automatically
# It reads the CF metadata, detects the 0-360 range, shifts it to -180 to 180, and sorts it.
ds = xc.swap_lon_axis(ds, to=(-180, 180))

/home/conda/dgiles/bb035d3e-1784479749-377-data_access/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
print(ds)


<xarray.Dataset> Size: 16GB
Dimensions:           (time: 304, orbit: 2, lat: 1800, lon: 3600, band: 2)
Coordinates:
  * time              (time) datetime64[ns] 2kB 2018-09-01 ... 2019-07-01
  * orbit             (orbit) object 16B 'Ascending' 'Descending'
  * lat               (lat) float32 7kB 89.95 89.85 89.75 ... -89.85 -89.95
  * lon               (lon) float32 14kB -180.0 -179.9 -179.8 ... 179.9 180.0
  * band              (band) object 16B 'snow_depth' 'quality_flag'
Data variables:
    geophysical_data  (time, orbit, lat, lon, band) int16 16GB dask.array<chunksize=(1, 1, 72, 3600, 2), meta=np.ndarray>


In [3]:
print(ds.geophysical_data)

<xarray.DataArray 'geophysical_data' (time: 304, orbit: 2, lat: 1800,
                                      lon: 3600, band: 2)> Size: 16GB
dask.array<getitem, shape=(304, 2, 1800, 3600, 2), dtype=int16, chunksize=(1, 1, 72, 3600, 2), chunktype=numpy.ndarray>
Coordinates:
  * time     (time) datetime64[ns] 2kB 2018-09-01 2018-09-02 ... 2019-07-01
  * orbit    (orbit) object 16B 'Ascending' 'Descending'
  * lat      (lat) float32 7kB 89.95 89.85 89.75 89.65 ... -89.75 -89.85 -89.95
  * lon      (lon) float32 14kB -180.0 -179.9 -179.8 ... 179.8 179.9 180.0
  * band     (band) object 16B 'snow_depth' 'quality_flag'
Attributes:
    long_name:      geophysical data (band 0 = snow depth, band 1 = quality f...
    units:          cm
    scale_factor:   0.1
    _FillValue:     -32768
    missing_value:  -32767


In [6]:
sub = ds.sel(lat=42.5, lon=-110.0, method="nearest").sel(
    orbit="Descending", band="snow_depth", time=slice("2019-01-01", "2019-02-01"))

print(sub["geophysical_data"].values)

[   336    320    343 -32767    329 -32767    323    324    324    334
    353    394 -32767    462    440    375    393    336    374 -32767
    365 -32767    380    390    345    345    386    389 -32767    429
    415    473]


In [7]:
sub = ds.sel(lat=42.5, lon=-110.0, method="nearest").sel(
    orbit="Descending", band="snow_depth", time=slice("2020-01-01", "2020-02-01"))

print(sub["geophysical_data"].values)

[]
